# The Weekend Effect — a quantitative teardown 🔬
### Real total-return tape · per-weekday HAC inference · five-test selection · pre/post-2000 difference

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Weekend effect still there?: Busted](https://img.shields.io/badge/Weekend_effect_still_there%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We separate the three things the folklore asserts — **Monday is negative**, **Tuesday beats the rest**, and **a calendar rule beats the index** — and find the first is false, the second weak, the third a mirage.

> ⚠️ **Not investment advice.** SPY daily, total-return adjusted (`quantlab.data`, Yahoo); cash earns 0% (conservative); 1 bp/switch; day-of-week is calendar-known so there is **no execution lag**. Sources in [`docs/references.md`](../docs/references.md), reproducible run in [`docs/results.md`](../docs/results.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # study root (weekend/)
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from weekend import data, strategy

AS_OF = "2026-06-12"
frame = data.load_real("SPY", mode="total_return").loc[:AS_OF]
close = frame["close"]
wm = strategy.weekday_means(close)
bh = strategy.buy_and_hold(close)
tue_only = strategy.backtest(close, strategy.tuesday_only_position(close), cost_bps=1.0)
skip_mon = strategy.backtest(close, strategy.skip_monday_position(close), cost_bps=1.0)
print(f"SPY total return: {len(close):,} rows  {close.index[0].date()} -> {close.index[-1].date()}  fingerprint={data.fingerprint(frame)}")
print(wm.round(2))


SPY total return: 8,400 rows  1993-01-29 -> 2026-06-12  fingerprint=c68e7ea2802e
     mean_bps         n  hac_t
Mon     5.640 1,578.000  2.060
Tue     7.170 1,725.000  2.640
Wed     6.200 1,723.000  2.210
Thu     1.460 1,690.000  0.520
Fri     3.320 1,683.000  1.330


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| **Signal** | `WEAK` | Tuesday's *own* mean clears the bar (+7.17 bps, HAC *t* +2.64), but the *claim* (Tuesday − rest) is +3.03 bps, HAC *t* **+1.00** (\|t\| < 2); the Monday claim is the wrong sign. Five weekday tests in play — no contrast survives a snooping-aware read. |
| **Tradability** | `MIRAGE` | Buy-Tuesday **2.36%** CAGR vs B&H **10.82%** (-8.46 pts/yr); Skip-Monday 7.3% (-3.52 pts/yr). Both Sharpes below B&H. The tilt is dwarfed by lost beta. |
| **Weekend effect still there?** | `BUSTED` | Monday − rest **+1.08 bps** (positive!), pre-2000 +4.78 and post-2000 +0.09; the change is −4.69 bps, HAC *t* -0.77. No negative Monday to find. |

> 💡 **In plain words:** the one part of the slogan that's directionally true (Tuesday is best) is too weak to bank, and the headline part (down-Monday) isn't true on this tape at all.

## 1 · The claim, steelmanned

- **H₁ (weekend effect):** Monday's mean return is significantly *negative* vs the rest.
- **H₂ (turnaround Tuesday):** Tuesday's mean return is significantly *higher* than the rest.
- **H₃ (tradable):** a literal weekday rule beats buy-and-hold on return / risk-adjusted return.

French (1980) found H₁ on 1953–1977; the modern literature reports it decayed. SPY starts in 1993, so this is squarely a *post-publication* sample.

## 2 · So what? — what rides on each answer

If H₃ held, the day of the week would dominate stock selection. It doesn't. And the *disappearance* of a once-real H₁ is a clean case study in how publishing an anomaly arbitrages it away — the more interesting finding than a fresh 'edge.'

## 3 · How we'd know — the protocol

Per-weekday mean with **HAC** *t* (daily returns are mildly autocorrelated) · the two contrasts as differences of means with HAC SEs combined in quadrature · a **pre-2000 vs post-2000** split with an explicit **test of the change** (not two separate means) on the pre-registered millennium cut · literal timers net of cost. **Mirage trigger:** no contrast clears |*t*| = 2 and every rule trails buy-and-hold.

## 4 · The teardown

Per-weekday means with HAC *t*-stats:

In [2]:
wm.round(3)

,mean_bps,n,hac_t
Mon,5.638,"1,578.000",2.061
Tue,7.172,"1,725.000",2.637
Wed,6.198,"1,723.000",2.209
Thu,1.457,"1,690.000",0.518
Fri,3.321,"1,683.000",1.326


The two headline **contrasts** — each weekday vs all the others, HAC *t* on the difference:

In [3]:
mon = strategy.contrast(close, 0)
tue = strategy.contrast(close, 1)
print(f"Monday  - rest: {mon['diff_bps']:+.2f} bps   HAC t {mon['hac_t']:+.2f}")
print(f"Tuesday - rest: {tue['diff_bps']:+.2f} bps   HAC t {tue['hac_t']:+.2f}")
print('=> Monday claim has the wrong sign; Tuesday claim is right-signed but |t| < 2.')

Monday  - rest: +1.08 bps   HAC t +0.36
Tuesday - rest: +3.03 bps   HAC t +1.00
=> Monday claim has the wrong sign; Tuesday claim is right-signed but |t| < 2.


> 💡 **In plain words:** Tuesday genuinely is the best day, but 'best of five' is significant by construction unless you correct for running five tests — and the honest difference-from-the-rest doesn't clear the bar.

**The decay story** — the *change* in each effect across the 2000 cut, with its own HAC *t*:

In [4]:
for wd, name in [(0,'Monday'),(1,'Tuesday')]:
    sp = strategy.subperiod_effect(close, wd, cut='2000-01-01')
    print(f"{name:8s} pre {sp['pre_diff_bps']:+.2f}  post {sp['post_diff_bps']:+.2f}  "
          f"change {sp['change_bps']:+.2f} bps  HAC t(change) {sp['hac_t_change']:+.2f}")
print('=> Monday is positive in BOTH halves; the change is not significant. Nothing decayed because nothing was there.')

Monday   pre +4.78  post +0.09  change -4.69 bps  HAC t(change) -0.77
Tuesday  pre +1.77  post +3.36  change +1.59 bps  HAC t(change) +0.23
=> Monday is positive in BOTH halves; the change is not significant. Nothing decayed because nothing was there.


> 💡 **In plain words:** you can't certify a 'decay' on this tape — there was no significant down-Monday in the first half to decay from. The negative weekend effect predates the SPY era entirely.

**The literal timers**, net of 1 bp/switch — the tradability read:

In [5]:
tbl = pd.DataFrame({
  'CAGR %':  [tue_only['cagr']*100, skip_mon['cagr']*100, bh['cagr']*100],
  'Vol %':   [tue_only['vol']*100, skip_mon['vol']*100, bh['vol']*100],
  'Sharpe':  [tue_only['sharpe'], skip_mon['sharpe'], bh['sharpe']],
  'MaxDD %': [tue_only['max_dd']*100, skip_mon['max_dd']*100, bh['max_dd']*100],
  'TiM %':   [tue_only['time_in_market']*100, skip_mon['time_in_market']*100, bh['time_in_market']*100],
}, index=['Buy Tuesday only', 'Skip Monday', 'Buy & hold'])
tbl.round(2)

,CAGR %,Vol %,Sharpe,MaxDD %,TiM %
Buy Tuesday only,2.360,8.370,0.320,-37.440,20.540
Skip Monday,7.300,16.360,0.510,-48.940,81.210
Buy & hold,10.820,18.580,0.650,-55.190,100.000


> 💡 **In plain words:** the lost equity premium on the sat-out days swamps any weekday tilt. This is the same lesson as the death-cross study — *less time in market is less return*, dressed up as a calendar edge.

## 5 · The verdict

Signal `WEAK` (Tuesday − rest +3.03 bps, HAC *t* +1.00; Monday wrong-signed), Tradability `MIRAGE` (buy-Tuesday -8.46 pts/yr, skip-Monday -3.52 pts/yr vs B&H), Weekend effect still there? `BUSTED` (Monday +1.08 bps, change HAC *t* -0.77). The literature's 1980 effect is not certifiable on the 1993–2026 SPY tape.

## 6 · Could you trade it?

Capacity is a non-issue (SPY). The binding fact is economic: any weekday filter forfeits the equity premium on the days it sits out, and that lost beta is an order of magnitude larger than the ~3 bps/day Tuesday tilt. Net of tax on thousands of round-trips the gap only widens. There is no residual edge to size.

## 7 · Going further

- Extend to **price-only `^GSPC`** back to the 1950s (label it price-only — no dividends) to test French's *original* window and date the effect's disappearance.
- Apply a **data-snooping correction** (White Reality Check / stationary bootstrap) over the five weekday rules to put a snooping-aware *p* on 'best weekday'.
- **Conditional** weekend effect: Monday return given a down Friday, with a Wilson interval on the conditional mean — does the unconditional flat hide a state-dependent tilt?